## 2.1卷积层参数

---

### 1. 输出特征图尺寸计算
#### 核心公式
卷积层输出特征图的**空间尺寸（高/宽）**计算公式：
$$
O = \left\lfloor \frac{I - K + 2P}{S} \right\rfloor + 1
$$
参数说明：
- $I$：输入特征图的空间尺寸（高/宽）
- $K$：卷积核的空间尺寸（高/宽）
- $P$：填充（Padding）大小
- $S$：步幅（Stride）
- 输出通道数 = 卷积核的总数量

#### 代入已知条件
已知输入：$3 \times 32 \times 32$（通道数×高×宽），卷积核：16个 $3 \times 5 \times 5$，$P=2$，$S=2$

1. **输出高/宽计算**：
$$
O = \left\lfloor \frac{32 - 5 + 2 \times 2}{2} \right\rfloor + 1 = \left\lfloor \frac{31}{2} \right\rfloor + 1 = 15 + 1 = 16
$$
2. **输出通道数**：等于卷积核数量，即16

#### 结论
输出特征图尺寸：$\boldsymbol{16 \times 16 \times 16}$（通道数 × 高 × 宽）

---

### 2. 单个输出像素的乘法次数计算
#### 原理
输出特征图中**单个像素的值**，是卷积核与输入对应感受野区域的所有元素**逐元素点乘后求和**的结果，乘法次数等于卷积核的总元素个数。

#### 计算过程
每个卷积核的尺寸为 $3 \times 5 \times 5$（输入通道数 × 卷积核高 × 卷积核宽），总元素个数（即乘法次数）：
$$
3 \times 5 \times 5 = 75
$$

#### 结论
单个输出通道的一个像素值，需要对输入进行 $\boldsymbol{75}$ 次乘法操作。

In [4]:
import numpy as np
import torch

def max_pool2d_manual_numpy(input, kernel_size, stride=None, padding=0):
    """NumPy 实现"""
    if input.ndim == 3:
        input = input[np.newaxis, ...]
        squeezed_batch = True
    else:
        squeezed_batch = False
    
    batch_size, channels, h_in, w_in = input.shape
    
    if isinstance(kernel_size, int):
        kh, kw = kernel_size, kernel_size
    else:
        kh, kw = kernel_size
    
    if stride is None:
        sh, sw = kh, kw
    elif isinstance(stride, int):
        sh, sw = stride, stride
    else:
        sh, sw = stride
    
    if isinstance(padding, int):
        ph, pw = padding, padding
    else:
        ph, pw = padding
    
    h_out = (h_in + 2 * ph - kh) // sh + 1
    w_out = (w_in + 2 * pw - kw) // sw + 1
    
    padded_input = np.pad(
        input,
        pad_width=((0, 0), (0, 0), (ph, ph), (pw, pw)),
        mode='constant',
        constant_values=-np.inf
    )
    
    output = np.zeros((batch_size, channels, h_out, w_out))
    
    for i in range(batch_size):
        for c in range(channels):
            for h in range(h_out):
                h_start = h * sh
                h_end = h_start + kh
                for w in range(w_out):
                    w_start = w * sw
                    w_end = w_start + kw
                    window = padded_input[i, c, h_start:h_end, w_start:w_end]
                    output[i, c, h, w] = np.max(window)
    
    return output[0] if squeezed_batch else output


def max_pool2d_manual_torch(input, kernel_size, stride=None, padding=0):
    """PyTorch 实现（不使用 nn.MaxPool2d）"""
    if isinstance(kernel_size, int):
        kh, kw = kernel_size, kernel_size
    else:
        kh, kw = kernel_size
    
    if stride is None:
        sh, sw = kh, kw
    elif isinstance(stride, int):
        sh, sw = stride, stride
    else:
        sh, sw = stride
    
    if isinstance(padding, int):
        ph, pw = padding, padding
    else:
        ph, pw = padding
    
    batch_size, channels, h_in, w_in = input.shape
    
    padded_input = torch.nn.functional.pad(
        input,
        pad=(pw, pw, ph, ph),
        mode='constant',
        value=float('-inf')
    )
    
    h_out = (h_in + 2 * ph - kh) // sh + 1
    w_out = (w_in + 2 * pw - kw) // sw + 1
    
    windows = torch.nn.functional.unfold(
        padded_input,
        kernel_size=(kh, kw),
        stride=(sh, sw)
    )
    
    windows = windows.view(batch_size, channels, kh * kw, h_out, w_out)
    output, _ = windows.max(dim=2)
    
    return output

## 3.1VGG网络卷积核参数量

---

### 前置公式：卷积层参数量计算（无偏置）
对于**不带偏置**的卷积层，参数量计算公式为：
$$
\text{参数量} = \text{输入通道数} \times \text{输出通道数} \times \text{卷积核高度} \times \text{卷积核宽度}
$$
> 原理：每个卷积核的尺寸为「输入通道数 × 核高 × 核宽」，总共有「输出通道数」个独立卷积核，无偏置则无需额外参数。

---

### 1. 单个 5×5 卷积层的参数量
#### 已知条件
- 输入通道数 = $C$，输出通道数 = $C$
- 卷积核尺寸 = $5 \times 5$
- 无偏置

#### 代入计算
$$
\text{参数量} = C \times C \times 5 \times 5 = 25C^2
$$

#### 结论
单个5×5卷积层的参数量为：$\boldsymbol{25C^2}$

---

### 2. 两个串联 3×3 卷积层的总参数量
#### 已知条件
- 两层卷积的输入、输出通道数均为 $C$
- 每层卷积核尺寸 = $3 \times 3$
- 两层均无偏置

#### 分步计算
1. **单个3×3卷积层的参数量**：
$$
\text{单层参数量} = C \times C \times 3 \times 3 = 9C^2
$$

2. **两层总参数量**：
$$
\text{总参数量} = 9C^2 + 9C^2 = 18C^2
$$

#### 结论
两个串联3×3卷积层的总参数量为：$\boldsymbol{18C^2}$

---

### 补充结论
两个3×3卷积级联可以实现与单个5×5卷积**相同的感受野**，但参数量仅为后者的 $\frac{18}{25}=72\%$，这也是VGG网络使用小卷积核级联的核心优势之一。

In [5]:
import torch
import torch.nn as nn

class NiNBlock(nn.Module):
    """
    NiN 块 (Network in Network Block)
    
    结构：
        普通卷积层 (kernel_size x kernel_size) -> ReLU
        -> 1x1 卷积层 -> ReLU
        -> 1x1 卷积层 -> ReLU
    
    参数：
        in_channels: 输入通道数
        out_channels: 输出通道数
        kernel_size: 中间卷积层的核大小
        stride: 中间卷积层的步幅
        padding: 中间卷积层的填充
    """
    def __init__(self, in_channels, out_channels, kernel_size, stride, padding):
        super(NiNBlock, self).__init__()
        
        self.block = nn.Sequential(
            # 第一层：普通卷积层
            nn.Conv2d(in_channels, out_channels, kernel_size, stride, padding),
            nn.ReLU(inplace=True),
            
            # 第二层：1x1 卷积层（相当于全连接层）
            nn.Conv2d(out_channels, out_channels, kernel_size=1),
            nn.ReLU(inplace=True),
            
            # 第三层：1x1 卷积层（相当于全连接层）
            nn.Conv2d(out_channels, out_channels, kernel_size=1),
            nn.ReLU(inplace=True)
        )
    
    def forward(self, x):
        return self.block(x)


# 更简洁的写法（使用函数式风格）
def nin_block(in_channels, out_channels, kernel_size, stride, padding):
    """
    返回一个 NiN 块的 Sequential 容器
    """
    return nn.Sequential(
        nn.Conv2d(in_channels, out_channels, kernel_size, stride, padding),
        nn.ReLU(inplace=True),
        nn.Conv2d(out_channels, out_channels, kernel_size=1),
        nn.ReLU(inplace=True),
        nn.Conv2d(out_channels, out_channels, kernel_size=1),
        nn.ReLU(inplace=True)
    )


# 使用示例
if __name__ == "__main__":
    # 示例1：创建 NiN 块
    block = NiNBlock(in_channels=3, out_channels=96, kernel_size=11, stride=4, padding=0)
    
    # 测试输入
    x = torch.randn(1, 3, 224, 224)
    out = block(x)
    
    print(f"输入形状: {x.shape}")
    print(f"输出形状: {out.shape}")
    print(f"\nNiN Block 结构:")
    print(block)
    
    # 示例2：使用函数式创建
    block2 = nin_block(96, 256, kernel_size=5, stride=1, padding=2)
    x2 = torch.randn(1, 96, 54, 54)
    out2 = block2(x2)
    
    print(f"\n输入形状: {x2.shape}")
    print(f"输出形状: {out2.shape}")

输入形状: torch.Size([1, 3, 224, 224])
输出形状: torch.Size([1, 96, 54, 54])

NiN Block 结构:
NiNBlock(
  (block): Sequential(
    (0): Conv2d(3, 96, kernel_size=(11, 11), stride=(4, 4))
    (1): ReLU(inplace=True)
    (2): Conv2d(96, 96, kernel_size=(1, 1), stride=(1, 1))
    (3): ReLU(inplace=True)
    (4): Conv2d(96, 96, kernel_size=(1, 1), stride=(1, 1))
    (5): ReLU(inplace=True)
  )
)

输入形状: torch.Size([1, 96, 54, 54])
输出形状: torch.Size([1, 256, 54, 54])


## 4.1Batch Normalization 

---

### Batch Normalization 核心计算流程
BN层的完整计算分为4步：
1. **计算批量均值**：
$$
\mu_B = \frac{1}{m}\sum_{i=1}^m x_i
$$
2. **计算批量方差**：
$$
\sigma_B^2 = \frac{1}{m}\sum_{i=1}^m (x_i - \mu_B)^2
$$
3. **特征标准化**：
$$
\hat{x}_i = \frac{x_i - \mu_B}{\sqrt{\sigma_B^2 + \epsilon}}
$$
4. **缩放与平移（可学习参数）**：
$$
y_i = \gamma \cdot \hat{x}_i + \beta
$$
其中：$m$为批量大小，$\gamma$为缩放参数，$\beta$为平移参数，$\epsilon$为防止除0的常数。

---

### 代入已知条件计算
已知：
- 批量样本值：$x_1=2,\ x_2=4,\ x_3=6,\ x_4=8$，批量大小 $m=4$
- 学习参数：$\gamma=2$，$\beta=1$，常数 $\epsilon=0$

---

#### 步骤1：计算批量均值
$$
\mu_B = \frac{x_1+x_2+x_3+x_4}{4} = \frac{2+4+6+8}{4} = 5
$$

#### 步骤2：计算批量方差
$$
\begin{align*}
\sigma_B^2 &= \frac{1}{4}\left[(2-5)^2 + (4-5)^2 + (6-5)^2 + (8-5)^2\right] \\
&= \frac{1}{4}\left[9 + 1 + 1 + 9\right] = \frac{20}{4} = 5
\end{align*}
$$
标准差：$\sqrt{\sigma_B^2 + \epsilon} = \sqrt{5+0} = \sqrt{5}$

#### 步骤3：特征标准化
$$
\begin{align*}
\hat{x}_1 &= \frac{2-5}{\sqrt{5}} = -\frac{3}{\sqrt{5}} \\
\hat{x}_2 &= \frac{4-5}{\sqrt{5}} = -\frac{1}{\sqrt{5}} \\
\hat{x}_3 &= \frac{6-5}{\sqrt{5}} = \frac{1}{\sqrt{5}} \\
\hat{x}_4 &= \frac{8-5}{\sqrt{5}} = \frac{3}{\sqrt{5}}
\end{align*}
$$

#### 步骤4：缩放平移得到最终输出
$$
\begin{align*}
y_1 &= 2 \times \left(-\frac{3}{\sqrt{5}}\right) + 1 = 1 - \frac{6\sqrt{5}}{5} \approx -1.683 \\
y_2 &= 2 \times \left(-\frac{1}{\sqrt{5}}\right) + 1 = 1 - \frac{2\sqrt{5}}{5} \approx 0.106 \\
y_3 &= 2 \times \frac{1}{\sqrt{5}} + 1 = 1 + \frac{2\sqrt{5}}{5} \approx 1.894 \\
y_4 &= 2 \times \frac{3}{\sqrt{5}} + 1 = 1 + \frac{6\sqrt{5}}{5} \approx 3.683
\end{align*}
$$

---

### 最终结果
$$
\boldsymbol{y_1 \approx -1.683,\ y_2 \approx 0.106,\ y_3 \approx 1.894,\ y_4 \approx 3.683}
$$
（精确表达式：$y_1=1-\frac{6\sqrt{5}}{5},\ y_2=1-\frac{2\sqrt{5}}{5},\ y_3=1+\frac{2\sqrt{5}}{5},\ y_4=1+\frac{6\sqrt{5}}{5}$）

In [6]:
import torch
import torch.nn as nn

class Residual(nn.Module):
    """
    残差块 (Residual Block)
    
    结构：
        3x3卷积 -> BN -> ReLU -> 3x3卷积 -> BN -> (+) -> ReLU
                                          ↑
                                          └── 残差连接（可选1x1卷积调整）
    
    参数：
        in_channels: 输入通道数
        out_channels: 输出通道数
        stride: 第一个卷积层的步幅（默认为1）
        use_1x1conv: 是否使用1x1卷积调整残差分支（默认False）
    """
    def __init__(self, in_channels, out_channels, stride=1, use_1x1conv=False):
        super(Residual, self).__init__()
        
        # 第一个卷积层：3x3，步幅可变
        self.conv1 = nn.Conv2d(in_channels, out_channels, 
                               kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        
        # 第二个卷积层：3x3，步幅固定为1
        self.conv2 = nn.Conv2d(out_channels, out_channels, 
                               kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        
        # ReLU 激活函数
        self.relu = nn.ReLU(inplace=True)
        
        # 残差分支：如果输入输出通道数不同或步幅不为1，使用1x1卷积调整
        if use_1x1conv:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, 
                         kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels)
            )
        else:
            # 如果不需要调整，直接使用恒等映射
            self.shortcut = nn.Identity()
    
    def forward(self, x):
        # 主路径：卷积 -> BN -> ReLU -> 卷积 -> BN
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)
        
        out = self.conv2(out)
        out = self.bn2(out)
        
        # 残差连接：主路径输出 + 短路径输出
        identity = self.shortcut(x)
        out += identity
        
        # 最后接 ReLU
        out = self.relu(out)
        
        return out


# 更详细的版本（带注释和多种残差块变体）
class ResidualBlock(nn.Module):
    """
    标准残差块（与 ResNet-18/34 相同）
    """
    def __init__(self, in_channels, out_channels, stride=1, downsample=None):
        super(ResidualBlock, self).__init__()
        
        self.conv1 = nn.Conv2d(in_channels, out_channels, 
                               kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        
        self.conv2 = nn.Conv2d(out_channels, out_channels, 
                               kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        
        self.relu = nn.ReLU(inplace=True)
        
        # downsample 用于调整残差分支
        self.downsample = downsample
    
    def forward(self, x):
        identity = x
        
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)
        
        out = self.conv2(out)
        out = self.bn2(out)
        
        # 如果 downsample 存在，对输入进行下采样
        if self.downsample is not None:
            identity = self.downsample(x)
        
        out += identity
        out = self.relu(out)
        
        return out


# 使用示例
if __name__ == "__main__":
    print("=" * 50)
    print("测试1：输入输出通道相同，步幅为1")
    print("=" * 50)
    
    # 情况1：不需要调整通道数
    block1 = Residual(in_channels=64, out_channels=64, stride=1, use_1x1conv=False)
    x1 = torch.randn(1, 64, 56, 56)
    out1 = block1(x1)
    print(f"输入形状: {x1.shape}")
    print(f"输出形状: {out1.shape}")
    print(f"参数量: {sum(p.numel() for p in block1.parameters()):,}")
    
    print("\n" + "=" * 50)
    print("测试2：输入输出通道不同，需要1x1卷积调整")
    print("=" * 50)
    
    # 情况2：需要调整通道数（如 ResNet 中增加通道数的地方）
    block2 = Residual(in_channels=64, out_channels=128, stride=2, use_1x1conv=True)
    x2 = torch.randn(1, 64, 56, 56)
    out2 = block2(x2)
    print(f"输入形状: {x2.shape}")
    print(f"输出形状: {out2.shape}")
    print(f"参数量: {sum(p.numel() for p in block2.parameters()):,}")
    
    print("\n" + "=" * 50)
    print("测试3：验证残差连接效果（梯度流）")
    print("=" * 50)
    
    # 测试梯度是否正常回传
    block3 = Residual(in_channels=32, out_channels=32, stride=1, use_1x1conv=False)
    x3 = torch.randn(1, 32, 32, 32, requires_grad=True)
    out3 = block3(x3)
    loss = out3.sum()
    loss.backward()
    
    print(f"输入梯度是否存在: {x3.grad is not None}")
    print(f"输入梯度范数: {x3.grad.norm().item():.6f}")
    print(f"卷积层1梯度范数: {block3.conv1.weight.grad.norm().item():.6f}")
    print(f"卷积层2梯度范数: {block3.conv2.weight.grad.norm().item():.6f}")
    
    print("\n" + "=" * 50)
    print("模型结构")
    print("=" * 50)
    print(block2)

测试1：输入输出通道相同，步幅为1
输入形状: torch.Size([1, 64, 56, 56])
输出形状: torch.Size([1, 64, 56, 56])
参数量: 73,984

测试2：输入输出通道不同，需要1x1卷积调整
输入形状: torch.Size([1, 64, 56, 56])
输出形状: torch.Size([1, 128, 28, 28])
参数量: 230,144

测试3：验证残差连接效果（梯度流）
输入梯度是否存在: True
输入梯度范数: 156.299454
卷积层1梯度范数: 3545.794434
卷积层2梯度范数: 3247.626221

模型结构
Residual(
  (conv1): Conv2d(64, 128, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
  (bn1): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (conv2): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
  (bn2): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (shortcut): Sequential(
    (0): Conv2d(64, 128, kernel_size=(1, 1), stride=(2, 2), bias=False)
    (1): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  )
)


## 5.1迁移学习微调（Fine-tuning）

---

### 问题1：底层特征层小学习率/冻结、顶层输出层大学习率的原因
#### 核心逻辑：不同网络层的特征通用性与训练需求存在本质差异
1. **底层特征提取层的特性**
    - 预训练网络的底层（卷积层、池化层等）学习到的是**通用底层视觉特征**：比如边缘、纹理、颜色、基础形状等，这类特征在绝大多数视觉任务中是通用的，在ImageNet这类大数据集上已经训练得非常成熟、鲁棒。
    - 若给底层设置大学习率，会破坏这些已经学好的通用特征，反而降低模型在目标任务上的性能；设置小学习率（或直接冻结）可以保留通用特征的迁移价值，仅做微小调整适配目标数据集。
    - 冻结底层还能大幅减少需要训练的参数量，降低小目标数据集下的过拟合风险。

2. **顶层输出层的特性**
    - 预训练网络的顶层（全连接输出层）是**针对源数据集的类别/任务专门设计的**：比如ImageNet的1000类分类头，和目标数据集的类别、任务完全不匹配，参数是针对源任务优化的，对目标任务没有价值。
    - 微调时顶层通常是**随机初始化**的，需要从零开始学习目标数据集的分类/回归逻辑，因此需要较大的学习率来快速收敛，适配目标任务的分布。

---

### 问题2：小样本、高相似性数据集的微调防过拟合策略
当目标数据集**规模极小**且**与源数据集分布高度相似**时，核心原则是：**最大程度复用预训练特征，最小化参数更新量**，具体策略如下：

1. **优先采用「特征提取+冻结全网络」策略**
    - 完全冻结所有预训练层的参数，仅将预训练网络作为**固定特征提取器**：输入目标数据集，输出网络倒数第二层的特征，用这些特征训练一个轻量级分类器（如逻辑回归、SVM、小尺寸全连接层）。
    - 优势：完全不更新预训练网络的参数，彻底避免破坏通用特征，且仅训练极少量的分类器参数，从根源上防止小样本过拟合。

2. **若必须微调部分层，仅解冻最顶部的1-2层特征层**
    - 仅解冻网络最顶部的1-2个高级特征层（靠近输出层的卷积/全连接层），其余所有底层完全冻结；且解冻层使用**极小的学习率**（通常是顶层学习率的1/10~1/100）。
    - 原理：最顶层的高级特征和任务相关性稍高，微小调整即可适配相似的目标数据集，底层通用特征完全不需要修改。

3. **叠加强正则化与数据增强**
    - 正则化：给可训练层增加L2权重衰减、Dropout层，限制参数的更新幅度；
    - 数据增强：对小样本目标数据集做高强度的数据增强（随机裁剪、翻转、颜色抖动、Mixup等），人为扩充数据的多样性，降低过拟合概率。

4. **早停（Early Stopping）**
    - 严格监控验证集的性能（准确率/损失），一旦验证集性能连续多轮不再提升，立即停止训练，避免模型在小训练集上过度拟合噪声。

In [7]:
import torch
import torchvision.transforms as transforms
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np

# 创建组合图像增广管道
data_transforms = transforms.Compose([
    # 1. 随机裁剪并缩放到 224x224
    # RandomResizedCrop 结合了随机裁剪和缩放，面积比例范围 [0.08, 1.0]
    transforms.RandomResizedCrop(size=224, scale=(0.08, 1.0)),
    
    # 2. 50% 概率水平翻转
    transforms.RandomHorizontalFlip(p=0.5),
    
    # 3. 随机改变亮度、对比度、饱和度
    # ColorJitter 参数: brightness, contrast, saturation, hue
    # 变化范围设为 0.5 意味着参数在 [0.5, 1.5] 之间随机选择
    transforms.ColorJitter(brightness=0.5, contrast=0.5, saturation=0.5),
    
    # 4. 转换为 PyTorch 张量
    # 将 PIL Image 或 numpy.ndarray (H x W x C) 转换为 Tensor (C x H x W)
    # 并自动将像素值缩放到 [0.0, 1.0]
    transforms.ToTensor()
])


# 更详细的版本（带参数说明）
def create_augmentation_pipeline(crop_size=224, 
                                   scale=(0.08, 1.0),
                                   hflip_prob=0.5,
                                   jitter_value=0.5):
    """
    创建图像增广管道
    
    参数：
        crop_size: 随机裁剪后的目标尺寸
        scale: 随机裁剪面积相对于原图的比例范围
        hflip_prob: 水平翻转的概率
        jitter_value: 颜色抖动的强度（亮度、对比度、饱和度）
    
    返回：
        transforms.Compose: 组合的增广管道
    """
    return transforms.Compose([
        transforms.RandomResizedCrop(size=crop_size, scale=scale),
        transforms.RandomHorizontalFlip(p=hflip_prob),
        transforms.ColorJitter(
            brightness=jitter_value,
            contrast=jitter_value,
            saturation=jitter_value
        ),
        transforms.ToTensor()
    ])


# 训练和验证时使用不同的管道（常见做法）
# 训练时使用增广，验证/测试时不使用随机增广
train_transforms = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.08, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.5, contrast=0.5, saturation=0.5),
    transforms.ToTensor(),
    # 可选：添加标准化（通常使用 ImageNet 的均值和标准差）
    # transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# 验证集只做确定性的预处理
val_transforms = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    # transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])


# 可视化示例
def visualize_augmentations(image_path, num_samples=5):
    """
    可视化图像增广效果
    
    参数：
        image_path: 图像路径
        num_samples: 显示多少个增广样本
    """
    # 加载原始图像
    original_image = Image.open(image_path).convert('RGB')
    
    # 创建图形
    fig, axes = plt.subplots(1, num_samples + 1, figsize=(15, 4))
    
    # 显示原始图像
    axes[0].imshow(original_image)
    axes[0].set_title('Original')
    axes[0].axis('off')
    
    # 显示增广后的图像
    for i in range(num_samples):
        # 每次应用相同的管道（但随机种子不同，产生不同结果）
        augmented = data_transforms(original_image)
        
        # 将张量转换回 numpy 用于显示
        # 注意：ToTensor 将值缩放到 [0,1]，需要转换回 [0,255]
        augmented_np = augmented.permute(1, 2, 0).numpy()
        
        axes[i + 1].imshow(augmented_np)
        axes[i + 1].set_title(f'Augmented {i+1}')
        axes[i + 1].axis('off')
    
    plt.tight_layout()
    plt.show()


# 使用示例
if __name__ == "__main__":
    print("图像增广管道配置：")
    print("=" * 50)
    print(data_transforms)
    
    print("\n" + "=" * 50)
    print("参数详解：")
    print("=" * 50)
    print("1. RandomResizedCrop(224, scale=(0.08, 1.0))")
    print("   - 从原图中随机裁剪一个区域，面积比例在 8% 到 100% 之间")
    print("   - 将裁剪区域缩放到 224x224 像素")
    print("   - 增强模型对不同尺度物体的适应性")
    
    print("\n2. RandomHorizontalFlip(p=0.5)")
    print("   - 50% 的概率水平翻转图像")
    print("   - 增强模型对左右对称的鲁棒性")
    
    print("\n3. ColorJitter(brightness=0.5, contrast=0.5, saturation=0.5)")
    print("   - 亮度：在 [0.5, 1.5] 范围内随机调整")
    print("   - 对比度：在 [0.5, 1.5] 范围内随机调整")
    print("   - 饱和度：在 [0.5, 1.5] 范围内随机调整")
    print("   - 增强模型对不同光照条件的适应性")
    
    print("\n4. ToTensor()")
    print("   - 将 PIL Image 或 numpy.ndarray 转换为 PyTorch 张量")
    print("   - 形状从 (H, W, C) 变为 (C, H, W)")
    print("   - 像素值从 [0, 255] 缩放到 [0.0, 1.0]")
    
    # 创建测试图像（如果没有真实图像，可以创建一个虚拟图像）
    print("\n" + "=" * 50)
    print("测试示例：")
    print("=" * 50)
    
    # 创建一个虚拟的测试图像（224x224 的随机彩色图像）
    dummy_image = Image.fromarray(np.random.randint(0, 255, (224, 224, 3), dtype=np.uint8))
    
    # 应用增广管道
    augmented_tensor = data_transforms(dummy_image)
    
    print(f"原始图像大小: {dummy_image.size}")
    print(f"增广后张量形状: {augmented_tensor.shape}")
    print(f"增广后张量值范围: [{augmented_tensor.min():.3f}, {augmented_tensor.max():.3f}]")
    
    # 如果有实际图像文件，可以取消下面的注释
    # image_path = "path/to/your/image.jpg"
    # visualize_augmentations(image_path, num_samples=4)
    
    print("\n" + "=" * 50)
    print("使用建议：")
    print("=" * 50)
    print("1. 训练集：使用完整的数据增广管道")
    print("2. 验证集：只使用 Resize + CenterCrop + ToTensor（不使用随机增广）")
    print("3. 测试集：与验证集相同，保证评估的稳定性")
    print("4. 根据任务特点调整增广参数：")
    print("   - 物体识别：更多使用几何变换")
    print("   - 细粒度分类：谨慎使用颜色变换")
    print("   - 医学图像：避免过强的几何变换")

图像增广管道配置：
Compose(
    RandomResizedCrop(size=(224, 224), scale=(0.08, 1.0), ratio=(0.75, 1.3333), interpolation=bilinear, antialias=warn)
    RandomHorizontalFlip(p=0.5)
    ColorJitter(brightness=(0.5, 1.5), contrast=(0.5, 1.5), saturation=(0.5, 1.5), hue=None)
    ToTensor()
)

参数详解：
1. RandomResizedCrop(224, scale=(0.08, 1.0))
   - 从原图中随机裁剪一个区域，面积比例在 8% 到 100% 之间
   - 将裁剪区域缩放到 224x224 像素
   - 增强模型对不同尺度物体的适应性

2. RandomHorizontalFlip(p=0.5)
   - 50% 的概率水平翻转图像
   - 增强模型对左右对称的鲁棒性

3. ColorJitter(brightness=0.5, contrast=0.5, saturation=0.5)
   - 亮度：在 [0.5, 1.5] 范围内随机调整
   - 对比度：在 [0.5, 1.5] 范围内随机调整
   - 饱和度：在 [0.5, 1.5] 范围内随机调整
   - 增强模型对不同光照条件的适应性

4. ToTensor()
   - 将 PIL Image 或 numpy.ndarray 转换为 PyTorch 张量
   - 形状从 (H, W, C) 变为 (C, H, W)
   - 像素值从 [0, 255] 缩放到 [0.0, 1.0]

测试示例：
原始图像大小: (224, 224)
增广后张量形状: torch.Size([3, 224, 224])
增广后张量值范围: [0.000, 1.000]

使用建议：
1. 训练集：使用完整的数据增广管道
2. 验证集：只使用 Resize + CenterCrop + ToTensor（不使用随机增广）
3. 测试集：与验证集相同，保证评估的稳定性
4. 根据任务特点调整增广参数：
   - 物体识别：

## 6.1目标检测交并比（IoU）

---

### IoU 核心计算公式
交并比（Intersection over Union）是预测框与真实框的**交集面积**与**并集面积**的比值：
$$
\text{IoU} = \frac{\text{Area}(A \cap B)}{\text{Area}(A \cup B)} = \frac{\text{交集面积}}{\text{真实框面积} + \text{预测框面积} - \text{交集面积}}
$$

边界框格式：`[x1, y1, x2, y2]`，其中 $(x1,y1)$ 为左上角坐标，$(x2,y2)$ 为右下角坐标。

---

### 代入已知条件计算
已知：
- 真实框 A：`[10, 10, 50, 50]`
- 预测框 B：`[30, 30, 70, 70]`

---

#### 步骤1：计算两个框的各自面积
1. **真实框A的面积**：
   宽度 = $x2 - x1 = 50 - 10 = 40$
   高度 = $y2 - y1 = 50 - 10 = 40$
   $$
   \text{Area}(A) = 40 \times 40 = 1600
   $$

2. **预测框B的面积**：
   宽度 = $70 - 30 = 40$
   高度 = $70 - 30 = 40$
   $$
   \text{Area}(B) = 40 \times 40 = 1600
   $$

---

#### 步骤2：计算交集区域的坐标与面积
交集区域的坐标规则：
- 交集左上角x = $\max(A.x1, B.x1)$
- 交集左上角y = $\max(A.y1, B.y1)$
- 交集右下角x = $\min(A.x2, B.x2)$
- 交集右下角y = $\min(A.y2, B.y2)$

代入计算：
$$
\begin{align*}
\text{交集}x1 &= \max(10, 30) = 30 \\
\text{交集}y1 &= \max(10, 30) = 30 \\
\text{交集}x2 &= \min(50, 70) = 50 \\
\text{交集}y2 &= \min(50, 70) = 50
\end{align*}
$$

交集的宽度与高度：
宽度 = $50 - 30 = 20$，高度 = $50 - 30 = 20$
$$
\text{Area}(A \cap B) = 20 \times 20 = 400
$$

---

#### 步骤3：计算并集面积与IoU
1. 并集面积：
$$
\text{Area}(A \cup B) = 1600 + 1600 - 400 = 2800
$$

2. 最终IoU：
$$
\text{IoU} = \frac{400}{2800} = \frac{1}{7} \approx 0.1429
$$

---

### 最终结果
边界框A与B的IoU准确值为 $\boldsymbol{\frac{1}{7}}$，近似值为 $\boldsymbol{0.1429}$。

In [8]:
import torch
import torch.nn as nn
import torch.nn.functional as F

def label_smoothing_cross_entropy(logits, targets, epsilon=0.1, reduction='mean'):
    """
    标签平滑交叉熵损失函数
    
    参数：
        logits: 模型的原始输出，形状 (batch_size, num_classes)
        targets: 真实标签，形状 (batch_size,)
        epsilon: 平滑因子，默认0.1
        reduction: 损失归约方式，'mean'、'sum' 或 'none'
    
    返回：
        损失值
    """
    num_classes = logits.size(-1)
    batch_size = logits.size(0)
    
    # 计算 log_softmax 以获得对数概率
    log_probs = F.log_softmax(logits, dim=-1)
    
    # 创建平滑后的标签分布
    # 真实类别的概率：1 - epsilon
    # 其他类别的概率：epsilon / (num_classes - 1)
    smooth_targets = torch.zeros_like(log_probs)
    smooth_targets.fill_(epsilon / (num_classes - 1))
    
    # 为真实类别设置概率 1 - epsilon
    smooth_targets.scatter_(1, targets.unsqueeze(1), 1 - epsilon)
    
    # 计算损失: -sum(smooth_targets * log_probs)
    loss = -torch.sum(smooth_targets * log_probs, dim=-1)
    
    # 根据 reduction 参数归约损失
    if reduction == 'mean':
        return loss.mean()
    elif reduction == 'sum':
        return loss.sum()
    else:  # 'none'
        return loss


class LabelSmoothingCrossEntropy(nn.Module):
    """
    标签平滑交叉熵损失模块
    
    可以直接在 nn.Module 中使用，与标准 CrossEntropyLoss 类似
    """
    def __init__(self, epsilon=0.1, reduction='mean'):
        super(LabelSmoothingCrossEntropy, self).__init__()
        self.epsilon = epsilon
        self.reduction = reduction
    
    def forward(self, logits, targets):
        return label_smoothing_cross_entropy(logits, targets, self.epsilon, self.reduction)


# 更高效的实现版本（避免显式创建 one-hot 矩阵）
def label_smoothing_cross_entropy_efficient(logits, targets, epsilon=0.1, reduction='mean'):
    """
    标签平滑交叉熵的高效实现
    通过分解损失公式，避免显式创建完整的 one-hot 矩阵
    """
    num_classes = logits.size(-1)
    
    # 计算 log_softmax
    log_probs = F.log_softmax(logits, dim=-1)
    
    # 计算标准交叉熵损失（不进行标签平滑）
    # 对于每个样本，取真实类别的负对数概率
    standard_loss = -log_probs.gather(dim=-1, index=targets.unsqueeze(1)).squeeze(1)
    
    # 计算均匀分布部分的损失
    # 均匀分布的概率为 1/num_classes
    uniform_loss = -log_probs.mean(dim=-1)
    
    # 组合损失
    # loss = (1 - epsilon) * standard_loss + epsilon * uniform_loss
    loss = (1 - epsilon) * standard_loss + epsilon * uniform_loss
    
    # 根据 reduction 参数归约损失
    if reduction == 'mean':
        return loss.mean()
    elif reduction == 'sum':
        return loss.sum()
    else:
        return loss


# 测试和验证
if __name__ == "__main__":
    # 设置随机种子以便重现
    torch.manual_seed(42)
    
    print("=" * 60)
    print("标签平滑交叉熵损失测试")
    print("=" * 60)
    
    # 测试参数
    batch_size = 4
    num_classes = 10
    epsilon = 0.1
    
    # 创建模拟数据
    logits = torch.randn(batch_size, num_classes)
    targets = torch.randint(0, num_classes, (batch_size,))
    
    print(f"批次大小: {batch_size}")
    print(f"类别数: {num_classes}")
    print(f"平滑因子 ε: {epsilon}")
    print(f"真实标签: {targets.tolist()}")
    
    print("\n" + "-" * 60)
    print("方法1: 显式创建平滑标签")
    print("-" * 60)
    
    loss1 = label_smoothing_cross_entropy(logits, targets, epsilon, reduction='mean')
    print(f"损失值: {loss1.item():.6f}")
    
    print("\n" + "-" * 60)
    print("方法2: 高效实现（分解公式）")
    print("-" * 60)
    
    loss2 = label_smoothing_cross_entropy_efficient(logits, targets, epsilon, reduction='mean')
    print(f"损失值: {loss2.item():.6f}")
    
    print(f"两种方法结果是否一致: {torch.allclose(loss1, loss2)}")
    
    print("\n" + "-" * 60)
    print("与标准交叉熵对比")
    print("-" * 60)
    
    # 标准交叉熵
    standard_ce = F.cross_entropy(logits, targets, reduction='mean')
    print(f"标准交叉熵损失: {standard_ce.item():.6f}")
    print(f"标签平滑损失: {loss1.item():.6f}")
    print(f"差异: {(loss1 - standard_ce).item():.6f}")
    
    print("\n" + "-" * 60)
    print("模块化使用示例")
    print("-" * 60)
    
    # 使用 nn.Module 版本
    criterion = LabelSmoothingCrossEntropy(epsilon=0.1, reduction='mean')
    loss_module = criterion(logits, targets)
    print(f"模块输出损失: {loss_module.item():.6f}")
    
    print("\n" + "-" * 60)
    print("不同平滑因子的影响")
    print("-" * 60)
    
    for eps in [0.0, 0.05, 0.1, 0.2, 0.5]:
        loss = label_smoothing_cross_entropy_efficient(logits, targets, epsilon=eps, reduction='mean')
        print(f"ε = {eps:<4.2f} -> 损失: {loss.item():.6f}")
    
    print("\n" + "-" * 60)
    print("详细计算示例（单个样本）")
    print("-" * 60)
    
    # 详细演示单个样本的计算过程
    sample_idx = 0
    sample_logits = logits[sample_idx:sample_idx+1]
    sample_target = targets[sample_idx]
    
    print(f"样本 {sample_idx}: 真实标签 = {sample_target}")
    print(f"Logits: {sample_logits.squeeze().tolist()}")
    
    # 计算概率分布
    probs = F.softmax(sample_logits, dim=-1).squeeze()
    print(f"Softmax 概率: {probs.tolist()}")
    
    # 创建平滑后的目标分布
    k = num_classes
    true_label_prob = 1 - epsilon
    other_label_prob = epsilon / (k - 1)
    
    smooth_target = torch.full((k,), other_label_prob)
    smooth_target[sample_target] = true_label_prob
    
    print(f"平滑后的目标分布: {smooth_target.tolist()}")
    
    # 计算交叉熵
    loss_sample = -torch.sum(smooth_target * torch.log(probs + 1e-8))
    print(f"单个样本损失: {loss_sample.item():.6f}")
    
    # 验证公式分解
    standard_loss_sample = -torch.log(probs[sample_target] + 1e-8)
    uniform_loss_sample = -torch.mean(torch.log(probs + 1e-8))
    decomposed_loss = (1 - epsilon) * standard_loss_sample + epsilon * uniform_loss_sample
    print(f"分解公式损失: {decomposed_loss.item():.6f}")
    print(f"两者一致: {torch.allclose(loss_sample, decomposed_loss)}")
    
    print("\n" + "=" * 60)
    print("关键数学公式")
    print("=" * 60)
    print("对于 K 分类问题，平滑因子 ε:")
    print(f"  q(k) = {{ 1-ε,         if k = y")
    print(f"          ε/(K-1),      otherwise }}")
    print("\n损失函数:")
    print("  L = -Σ q(k) * log(p(k))")
    print("    = (1-ε) * (-log(p(y))) + ε * (-1/K Σ log(p(k)))")

标签平滑交叉熵损失测试
批次大小: 4
类别数: 10
平滑因子 ε: 0.1
真实标签: [5, 4, 8, 8]

------------------------------------------------------------
方法1: 显式创建平滑标签
------------------------------------------------------------
损失值: 3.404533

------------------------------------------------------------
方法2: 高效实现（分解公式）
------------------------------------------------------------
损失值: 3.411288
两种方法结果是否一致: False

------------------------------------------------------------
与标准交叉熵对比
------------------------------------------------------------
标准交叉熵损失: 3.472086
标签平滑损失: 3.404533
差异: -0.067554

------------------------------------------------------------
模块化使用示例
------------------------------------------------------------
模块输出损失: 3.404533

------------------------------------------------------------
不同平滑因子的影响
------------------------------------------------------------
ε = 0.00 -> 损失: 3.472086
ε = 0.05 -> 损失: 3.441687
ε = 0.10 -> 损失: 3.411288
ε = 0.20 -> 损失: 3.350490
ε = 0.50 -> 损失: 3.168097

-------------------------------